## 시나리오

매번 새 회의록을 만들 때마다 같은 구조로 반복하는 작업을 자동화합니다.

1. **제목** (큰 글씨, 가운데 정렬)
2. **회의 일시** (자동으로 오늘 날짜)
3. **참석자 명단** (표 형태)
4. **안건** (번호 매긴 목록)
5. **결론** (굵은 제목 + 본문)

## 전체 코드

```python
import datetime
from hwpapi.core import App


def create_meeting_minutes(
    title: str,
    attendees: list[tuple[str, str]],   # [(이름, 소속), ...]
    agenda: list[str],
    conclusion: str,
    output_path: str,
):
    app = App(is_visible=True)

    # 1. 제목 — 가운데, 20pt, 굵게
    app.insert_text(title + "\n")
    app.move.top_of_file()
    app.select_text()
    app.set_charshape(bold=True, height=2000)
    app.set_parashape(align_type=3)  # center

    # 기본 서식으로 복귀
    app.move.bottom_of_file()
    app.set_charshape(bold=False, height=1000)
    app.set_parashape(align_type=1)

    # 2. 회의 일시
    today = datetime.datetime.now().strftime("%Y년 %m월 %d일")
    app.insert_text(f"\n□ 회의 일시: {today}\n\n")

    # 3. 참석자 표
    app.insert_text("□ 참석자\n")
    action = app.actions.TableCreate
    action.pset.Rows = len(attendees) + 1
    action.pset.Cols = 2
    action.run()

    # 헤더
    for header in ["이름", "소속"]:
        app.insert_text(header)
        app.api.Run("TableRightCell")
    for name, org in attendees:
        app.insert_text(name)
        app.api.Run("TableRightCell")
        app.insert_text(org)
        app.api.Run("TableRightCell")

    app.move.bottom_of_file()

    # 4. 안건 — 번호 매긴 목록
    app.insert_text("\n□ 안건\n")
    for i, item in enumerate(agenda, 1):
        app.insert_text(f"  {i}. {item}\n")

    # 5. 결론
    app.insert_text("\n□ 결론\n")
    app.set_charshape(bold=True)
    app.insert_text("● ")
    app.set_charshape(bold=False)
    app.insert_text(conclusion + "\n")

    # 저장
    app.save(output_path)
    print(f"회의록 저장 완료: {output_path}")
    return app


# 사용 예
create_meeting_minutes(
    title="2026년 1분기 정기회의",
    attendees=[
        ("김철수", "기획팀"),
        ("이영희", "개발팀"),
        ("박민수", "마케팅팀"),
    ],
    agenda=[
        "1분기 실적 검토",
        "2분기 신규 프로젝트 계획",
        "인력 충원 논의",
    ],
    conclusion="2분기 프로젝트 A/B/C 착수, 인력 2명 충원 확정",
    output_path="minutes_2026Q1.hwp",
)
```

## 핵심 포인트

- **서식 스택**: 제목용 서식 적용 후 본문용으로 되돌려야 이후 문단에 영향이 없습니다.
- **표 셀 이동**: `TableCreate` 직후 커서는 A1 셀에 있습니다. `app.api.Run("TableRightCell")` 로 다음 셀로 이동합니다.
- **표 밖으로**: 마지막 셀에서 한 번 더 `TableRightCell` 을 하면 표 다음 줄로 나옵니다. 또는 `app.move.bottom_of_file()` 사용.
- **날짜**: `datetime` 모듈로 직접 포맷팅하면 원하는 형태로 쉽게 쓸 수 있습니다. HWP의 `InsertStringDateTime` 액션을 쓰면 HWP 기본 포맷이 됩니다.

## 확장 아이디어

- 템플릿 YAML/JSON에서 회의 데이터를 읽어 여러 회의록을 일괄 생성
- 이전 회의록에서 이월 안건을 자동으로 복사
- PDF로도 동시 저장: `app.save("minutes.pdf")`

## 📄 실행 결과 문서

생성된 HWP 문서(`minutes_2026Q1.hwp`)는 이런 모습입니다:

<div class="preview-doc">

<div style="text-align:center;font-size:1.6rem;font-weight:800;margin:1.5rem 0 1rem;">2026년 1분기 정기회의</div>

<p style="margin:0.5rem 0">□ 회의 일시: 2026년 04월 15일</p>

<p style="margin:1rem 0 0.3rem">□ 참석자</p>

<table style="border-collapse:collapse;margin:0.5rem 0;">
<thead><tr><th style="border:1px solid #888;padding:0.4rem 1.2rem;background:#f4f4f4;">이름</th><th style="border:1px solid #888;padding:0.4rem 1.2rem;background:#f4f4f4;">소속</th></tr></thead>
<tbody>
<tr><td style="border:1px solid #888;padding:0.4rem 1.2rem;">김철수</td><td style="border:1px solid #888;padding:0.4rem 1.2rem;">기획팀</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1.2rem;">이영희</td><td style="border:1px solid #888;padding:0.4rem 1.2rem;">개발팀</td></tr>
<tr><td style="border:1px solid #888;padding:0.4rem 1.2rem;">박민수</td><td style="border:1px solid #888;padding:0.4rem 1.2rem;">마케팅팀</td></tr>
</tbody>
</table>

<p style="margin:1rem 0 0.3rem">□ 안건</p>
<ol style="margin:0;padding-left:2rem;">
<li>1분기 실적 검토</li>
<li>2분기 신규 프로젝트 계획</li>
<li>인력 충원 논의</li>
</ol>

<p style="margin:1rem 0 0.3rem">□ 결론</p>
<p style="margin:0;padding-left:1rem;"><strong>●</strong> 2분기 프로젝트 A/B/C 착수, 인력 2명 충원 확정</p>


**콘솔 출력:**

```
회의록 저장 완료: minutes_2026Q1.hwp
```